In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "data"))
sys.path.append(str(Path.cwd().parent / "bayesian_order_based_learning"))

In [2]:
import scipy.io
import pandas as pd
import numpy as np
import networkx as nx
import os
import re

In [3]:
data_path_rel = os.path.join(Path.cwd().parent, "data")

gene_matrix = scipy.io.mmread(os.path.join(data_path_rel, "GSE144136_GeneBarcodeMatrix_Annotated.mtx"))
gene_matrix = gene_matrix.tocsr()

gene_names = pd.read_csv(os.path.join(data_path_rel, "GSE144136_GeneNames.csv"))
gene_names.set_index('Unnamed: 0', inplace=True)
cell_names = pd.read_csv(os.path.join(data_path_rel, "GSE144136_CellNames.csv"))
cell_names.set_index('Unnamed: 0', inplace=True)

gene_df = pd.DataFrame.sparse.from_spmatrix(gene_matrix, index=gene_names["x"], columns=cell_names["x"])

In [4]:
gene_list = pd.read_excel(os.path.join(data_path_rel, "41593_2020_621_MOESM3_ESM.xlsx"), sheet_name="Supplementary Table 32", header=2)

top_41_genes = gene_list[gene_list["FDR"] < 0.1]
top_41_genes["idx"] = range(len(top_41_genes))

/var/folders/nv/0gm0y9v14g5442csxys2r93h0000gn/T/ipykernel_38365/2222079740.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  top_41_genes["idx"] = range(len(top_41_genes))


In [5]:
constrained_gene_df = gene_df[gene_df.index.isin(top_41_genes["Gene"])]
constrained_gene_df_idx = constrained_gene_df.rename(index=dict(zip(top_41_genes["Gene"], top_41_genes["idx"])))
constrained_gene_df_idx = constrained_gene_df_idx.sort_index(axis=0)

In [6]:
def get_patient(cell_name):
    return re.search(r'\.(\d+)_(?:Suicide|Control)_B\d+', cell_name).group(1)

patient_dfs = []

for patient_num, patient_df in constrained_gene_df_idx.T.groupby(get_patient):
    patient_dfs.append((int(patient_num), patient_df))

In [7]:
patient_dfs.sort()

In [8]:
len(patient_dfs)

34

In [9]:
# 24
patient_dfs[24][1]

x,0,1,2,3,4,5,6,7,8,9,...,31,32,33,34,35,36,37,38,39,40
x,,,,,,,,,,,,,,,,,,,,,
Astros_2.25_Control_B3_AAACGGGCAAATCCGT,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Astros_2.25_Control_B3_AAAGATGTCCATGCTC,0,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Ex_3_L4_5.25_Control_B3_AAGGCAGCATCGTCGG,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
Ex_10_L2_4.25_Control_B3_AAGTCTGAGCCACTAT,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Ex_10_L2_4.25_Control_B3_ACACTGACACTAAGTC,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Astros_2.25_Control_B5_TTCTTAGAGCCAACAG,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
Ex_10_L2_4.25_Control_B5_TTCTTAGAGTGGAGTC,0,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Ex_10_L2_4.25_Control_B5_TTGCCGTGTCTAGTCA,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [10]:
constrained_gene_df

x,Ex_10_L2_4.3_Control_B3_AAACCTGAGGTAGCCA,Inhib_5.3_Control_B3_AAACCTGCAAACTGTC,Ex_10_L2_4.3_Control_B3_AAACCTGCAACAACCT,Ex_8_L5_6.3_Control_B3_AAACCTGGTCCGAACC,Inhib_2_VIP.3_Control_B3_AAACCTGGTCGTGGCT,Mix_2.3_Control_B3_AAACGGGAGGAACTGC,Oligos_3.3_Control_B3_AAACGGGAGTGGTAGC,Ex_7_L4_6.3_Control_B3_AAACGGGAGTGTGAAT,Astros_3.3_Control_B3_AAACGGGCATCCAACA,Ex_5_L5.3_Control_B3_AAACGGGGTATTCGTG,...,Inhib_5.17_Suicide_B1_TTTGCGCTCGGCTTGG,Ex_10_L2_4.17_Suicide_B1_TTTGGTTCAGTCCTTC,Inhib_6_SST.17_Suicide_B1_TTTGGTTGTCCAGTGC,Ex_10_L2_4.17_Suicide_B1_TTTGGTTTCTGTCCGT,Micro/Macro.17_Suicide_B1_TTTGTCAAGATGTCGG,OPCs_1.17_Suicide_B1_TTTGTCAAGCTCCTTC,Ex_10_L2_4.17_Suicide_B1_TTTGTCAAGGCTAGCA,Ex_10_L2_4.17_Suicide_B1_TTTGTCAAGGGATGGG,Inhib_2_VIP.17_Suicide_B1_TTTGTCACATGTCCTC,Inhib_7_PVALB.17_Suicide_B1_TTTGTCAGTAAGTGGC
x,,,,,,,,,,,,,,,,,,,,,
KAZN,1,1,1,0,2,0,1,2,0,0,...,3,0,1,1,3,0,1,0,4,2
PINK1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
ARL8A,0,0,0,0,0,0,0,0,0,1,...,0,0,0,1,0,0,0,0,0,0
ADORA1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
KIF26B,0,0,1,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
AC133680.1,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,2,0,0,0,0,0
ACAA1,0,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
CAMK2N2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
HIVEP1,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,1,0,1,0,0,0


In [11]:
constrained_gene_df_idx

x,Ex_10_L2_4.3_Control_B3_AAACCTGAGGTAGCCA,Inhib_5.3_Control_B3_AAACCTGCAAACTGTC,Ex_10_L2_4.3_Control_B3_AAACCTGCAACAACCT,Ex_8_L5_6.3_Control_B3_AAACCTGGTCCGAACC,Inhib_2_VIP.3_Control_B3_AAACCTGGTCGTGGCT,Mix_2.3_Control_B3_AAACGGGAGGAACTGC,Oligos_3.3_Control_B3_AAACGGGAGTGGTAGC,Ex_7_L4_6.3_Control_B3_AAACGGGAGTGTGAAT,Astros_3.3_Control_B3_AAACGGGCATCCAACA,Ex_5_L5.3_Control_B3_AAACGGGGTATTCGTG,...,Inhib_5.17_Suicide_B1_TTTGCGCTCGGCTTGG,Ex_10_L2_4.17_Suicide_B1_TTTGGTTCAGTCCTTC,Inhib_6_SST.17_Suicide_B1_TTTGGTTGTCCAGTGC,Ex_10_L2_4.17_Suicide_B1_TTTGGTTTCTGTCCGT,Micro/Macro.17_Suicide_B1_TTTGTCAAGATGTCGG,OPCs_1.17_Suicide_B1_TTTGTCAAGCTCCTTC,Ex_10_L2_4.17_Suicide_B1_TTTGTCAAGGCTAGCA,Ex_10_L2_4.17_Suicide_B1_TTTGTCAAGGGATGGG,Inhib_2_VIP.17_Suicide_B1_TTTGTCACATGTCCTC,Inhib_7_PVALB.17_Suicide_B1_TTTGTCAGTAAGTGGC
x,,,,,,,,,,,,,,,,,,,,,
0,1,0,0,0,1,0,0,1,0,0,...,0,0,2,0,0,0,0,1,0,0
1,0,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1,1,1,0,2,0,1,2,0,0,...,3,0,1,1,3,0,1,0,4,2
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6,3,0,1,1,0,0,0,2,0,0,...,0,0,0,0,1,0,1,0,0,0
7,0,0,1,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,1
8,1,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [12]:
top_41_genes

,Gene,Estimate,Std. Error,df,t value,Pr(>|t|),Fold Change,FDR,Cluster,idx
0,TMSB4X,-0.401627,0.065789,646.000000,-6.104766,1.776243e-09,0.669231,0.000154,Astro3,0
1,ACAA1,-0.113096,0.018823,962.561266,-6.008367,2.655221e-09,0.893065,0.000154,Ex7,1
2,RPS11,-0.080013,0.014327,3368.000000,-5.584886,2.524009e-08,0.923105,0.000977,Ex7,2
3,KAZN,0.485059,0.095822,263.275306,5.062061,7.799085e-07,1.624271,0.021070,OPC2,3
4,PRAF2,-0.061566,0.012332,383.242170,-4.992326,9.074639e-07,0.940291,0.021070,Ex3,4
5,GRM4,-0.099405,0.020775,1625.000000,-4.784921,1.865614e-06,0.905376,0.029844,Ex8,5
6,NRXN2,-0.183301,0.038823,1434.951257,-4.721444,2.570690e-06,0.832518,0.029844,In1,6
7,TMEM132A,-0.106942,0.022638,2280.250645,-4.723975,2.453433e-06,0.898578,0.029844,In2,7
8,PTOV1,-0.109207,0.022719,2825.999973,-4.806804,1.613831e-06,0.896545,0.029844,In2,8
9,TCEAL2,-0.253221,0.052822,521.144045,-4.793852,2.137576e-06,0.776297,0.029844,In8,9


In [13]:
X = []
for _, pat_k in patient_dfs:
    X.append(pat_k.to_numpy())

In [14]:
X[0]

array([[2, 0, 0, ..., 0, 0, 0],
       [9, 1, 1, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [3, 0, 1, ..., 0, 0, 1],
       [0, 0, 0, ..., 0, 0, 0],
       [2, 0, 0, ..., 1, 0, 0]], shape=(3267, 41))

In [15]:
patient_dfs[0][1]

x,0,1,2,3,4,5,6,7,8,9,...,31,32,33,34,35,36,37,38,39,40
x,,,,,,,,,,,,,,,,,,,,,
Ex_2_L5.1_Suicide_B5_AAACCTGAGGACAGCT,2,0,0,0,0,0,0,2,1,0,...,0,1,1,2,0,1,3,0,0,0
Ex_3_L4_5.1_Suicide_B5_AAACCTGAGGCCCTTG,9,1,1,1,1,0,1,0,1,1,...,2,1,3,8,0,0,5,0,0,0
Inhib_1.1_Suicide_B5_AAACCTGCATTCCTCG,0,0,0,2,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
Ex_1_L5_6.1_Suicide_B5_AAACCTGGTCGAGTTT,2,0,0,0,0,0,0,0,1,1,...,0,0,0,1,1,0,0,1,0,1
Ex_3_L4_5.1_Suicide_B5_AAACCTGTCATGTCTT,3,0,0,1,0,0,3,0,0,1,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Ex_3_L4_5.1_Suicide_B5_TTTGTCAGTAGCAAAT,1,0,0,2,0,0,0,1,0,0,...,1,0,0,2,0,1,0,0,0,1
OPCs_1.1_Suicide_B5_TTTGTCAGTATGCTTG,1,0,0,1,0,0,0,0,0,0,...,0,0,0,1,0,1,0,0,0,0
Ex_3_L4_5.1_Suicide_B5_TTTGTCAGTGGCCCTA,3,0,1,0,0,0,1,0,0,0,...,0,0,1,2,1,0,0,0,0,1


### Order based learner applied to gene expression data

In [16]:
from bayesian_order_based_learner import OrderBasedLearner

/Users/sreehari_miniravi/Work/ResearchProjects/bayesian-multi-dag/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-08-16 04:02:39,366	INFO util.py:155 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [17]:
# Exclude the sample with limited data
X_33 = X[:24] + X[25:]

In [18]:
p = 41
sigma_0 = np.arange(0, p)
obl_gene_expression = OrderBasedLearner(X_33, sigma_0, T=30000, verbose=True)

2026-08-16 04:02:41,131	INFO worker.py:2024 -- Started a local Ray instance.
2026-08-16 04:02:41,135	WARNING working_dir.py:91 -- Directory '__pycache__' is now ignored by default when packaging the working directory. To disable this behavior, set the `RAY_OVERRIDE_RUNTIME_ENV_DEFAULT_EXCLUDES=''` environment variable.
2026-08-16 04:02:41,139	INFO packaging.py:734 -- Creating a file package for local module '/Users/sreehari_miniravi/Work/ResearchProjects/bayesian-multi-dag/bayesian_order_based_learning'.
2026-08-16 04:02:41,142	INFO packaging.py:434 -- Pushing file package 'gcs://_ray_pkg_657919d64db8a03b.zip' (0.01MiB) to Ray cluster...
2026-08-16 04:02:41,143	INFO packaging.py:447 -- Successfully pushed file package 'gcs://_ray_pkg_657919d64db8a03b.zip'.


In [19]:
orderings, dags, log_posteriors = obl_gene_expression.compute()
obl_gene_expression.shutdown()

MCMC Sampling wth R2R:   0%|          | 12/30000 [00:00<14:04, 35.50it/s]/Users/sreehari_miniravi/Work/ResearchProjects/bayesian-multi-dag/bayesian_order_based_learning/bayesian_order_based_learner.py:96: RuntimeWarning: overflow encountered in exp
  a = min(np.exp(pi_sigma_curr - pi_sigma_prev), 1)
MCMC Sampling wth R2R:   0%|          | 34/30000 [00:26<6:25:19,  1.30it/s] 


KeyboardInterrupt: 

In [21]:
import pickle

data = {
    "orderings": orderings,
    "dags": dags,
    "log_posteriors": log_posteriors
}

with open(os.path.join(data_path_rel, "gene_expression_results.pkl"), "wb") as f:
    pickle.dump(data, f)

NameError: name 'orderings' is not defined

In [22]:
data = {}
with open(os.path.join(data_path_rel, "gene_expression_results.pkl"), "rb") as f:
    data = pickle.load(f)

In [23]:
orderings = data["orderings"]
dags = data["dags"]
log_posteriors = data["log_posteriors"]

In [ ]:
i = 32

In [ ]:
X_33[i]

In [ ]:
patient_dfs[i+1][1]

### Evaluating genes

In [24]:
case_patient_idx = [0, 3, 4, 5, 7, 9, 10, 13, 16, 17, 22, 24, 26, 28, 30, 31, 32]
control_patient_idx = [1, 2, 6, 8, 11, 12, 14, 15, 18, 19, 20, 21, 23, 25, 27, 29]

In [25]:
from metrics import edge_probability

K = 33
p = 41

dags_markov_chain = [[] for _ in range(K)]
adj_matrices_markov_chain = [[] for _ in range(K)]
for dag_t in dags:
    for k, dag in enumerate(dag_t):
        dags_markov_chain[k].append(dag)
        adj_matrices_markov_chain[k].append(nx.to_numpy_array(dag, weight=None, dtype=np.dtype(int)))

connectivity_case = np.zeros(p)
connectivity_control = np.zeros(p)
for k in range(K):
    adj_matrices = np.array(adj_matrices_markov_chain[k])

    for node in range(p):
        connectivity = 0
        for other_node in range(p):
            if other_node == node:
                continue

            connectivity += edge_probability(adj_matrices, (node, other_node))
            connectivity += edge_probability(adj_matrices, (other_node, node))

        if k in case_patient_idx:
            connectivity_case[node] += connectivity
        else:
            connectivity_control[node] += connectivity

connectivity_case = connectivity_case / (len(case_patient_idx))
connectivity_control = connectivity_control / (len(control_patient_idx))

"""
for dag_t in dags:
    for k, dag in enumerate(dag_t):
        adj_matrices = np.array(adj_matrices_markov_chain[k])
        for node in range(p):
            for in_edge in dag.in_edges(node):
                if k in case_patient_idx:
                    connectivity_case[node] += edge_probability(adj_matrices, in_edge)
                else:
                    connectivity_control[node] += edge_probability(adj_matrices, in_edge)

            for out_edge in dag.out_edges(node):
                if k in case_patient_idx:
                    connectivity_case[node] += edge_probability(adj_matrices, out_edge)
                else:
                    connectivity_control[node] += edge_probability(adj_matrices, out_edge)

connectivity_case = connectivity_case / (len(case_patient_idx))
connectivity_control = connectivity_control / (len(control_patient_idx))
"""

'\nfor dag_t in dags:\n    for k, dag in enumerate(dag_t):\n        adj_matrices = np.array(adj_matrices_markov_chain[k])\n        for node in range(p):\n            for in_edge in dag.in_edges(node):\n                if k in case_patient_idx:\n                    connectivity_case[node] += edge_probability(adj_matrices, in_edge)\n                else:\n                    connectivity_control[node] += edge_probability(adj_matrices, in_edge)\n\n            for out_edge in dag.out_edges(node):\n                if k in case_patient_idx:\n                    connectivity_case[node] += edge_probability(adj_matrices, out_edge)\n                else:\n                    connectivity_control[node] += edge_probability(adj_matrices, out_edge)\n\nconnectivity_case = connectivity_case / (len(case_patient_idx))\nconnectivity_control = connectivity_control / (len(control_patient_idx))\n'

In [26]:
group_diff = connectivity_control - connectivity_case
idx_diff = {idx: group_diff[idx] for idx in range(len(group_diff))}
top_20 = dict(sorted(idx_diff.items(), key=lambda item: item[1], reverse=True)[:20])
top_9 = dict(sorted(idx_diff.items(), key=lambda item: item[1], reverse=True)[:9])

mapping_dict = top_41_genes.set_index('idx')["Gene"].to_dict()

top_20 = {
    mapping_dict[idx]: value
    for idx, value in top_20.items()
    if idx in mapping_dict
}

top_9 = {
    mapping_dict[idx]: value
    for idx, value in top_9.items()
    if idx in mapping_dict
}

In [27]:
top_20

{'TUBB4B': np.float64(0.6530713235294123),
 'RAB11B': np.float64(0.505650980392157),
 'ARL8A': np.float64(0.42762156862745115),
 'PRAF2': np.float64(0.3587394607843135),
 'PTOV1': np.float64(0.34925318627450963),
 'ACAA1': np.float64(0.3353183823529413),
 'NRXN2': np.float64(0.3123848039215691),
 'PPME1': np.float64(0.2778958333333339),
 'RPS11': np.float64(0.25330784313725463),
 'TAPBP': np.float64(0.24848186274509754),
 'PEBP1': np.float64(0.2439968137254871),
 'CAMK2N2': np.float64(0.23197524509803968),
 'TMEM132A': np.float64(0.2306749999999993),
 'ADORA1': np.float64(0.20590367647058772),
 'KLC2': np.float64(0.19761372549019507),
 'ALDOA': np.float64(0.18006593137255011),
 'PINK1': np.float64(0.17005882352941182),
 'MAP2K7': np.float64(0.1237051470588244),
 'CCDC130': np.float64(0.1070301470588233),
 'CEND1': np.float64(0.10170710784313819)}

In [28]:
top_9

{'TUBB4B': np.float64(0.6530713235294123),
 'RAB11B': np.float64(0.505650980392157),
 'ARL8A': np.float64(0.42762156862745115),
 'PRAF2': np.float64(0.3587394607843135),
 'PTOV1': np.float64(0.34925318627450963),
 'ACAA1': np.float64(0.3353183823529413),
 'NRXN2': np.float64(0.3123848039215691),
 'PPME1': np.float64(0.2778958333333339),
 'RPS11': np.float64(0.25330784313725463)}

In [29]:
top_9_paper = ["GRIN2A", "CCND1", "PRKAR1B", "PEBP1", "HSP90AA1", "NRXN2", "ALDOA", "TMSB4X", "TOMIL1"]

top_9_common = {
    idx: value
    for idx, value in top_9.items()
    if idx in top_9_paper
}

In [30]:
top_9_common

{'NRXN2': np.float64(0.3123848039215691)}